<a href="https://colab.research.google.com/github/DhavalRepo18/Agent-Oriented-Planning/blob/main/notebook/LLM_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Hands-on Notebook

This notebook demonstrates how to use LangChain with a Watsonx.ai Large Language Model (LLM) and how to create and use a custom tool with LLM agents. Some key concepts:

- **LLM (Large Language Model):** A type of artificial intelligence that can understand and generate human-like text.
- **LangChain:** A popular python framework for developing applications powered by language models.
- **LLM Agent:** An entity that uses an LLM to decide which actions to take, from a set of available tools, to achieve a goal.

What we'll cover:

* Understanding what LLM agents are and why they matter
* Setting up LangChain with IBM Watsonx.ai
* Building agents that can use tools to solve real problems
* Practical examples you can implement immediately

## Install langchain with Watsonx.ai

In [4]:
!pip install -U langchain-classic langchain-ibm langchain-community langchain-core

In [5]:
import langchain
from langchain_ibm import WatsonxLLM

In [6]:
langchain.__version__

'1.2.12'

## Load Watsonx.ai credentials
Watsonx.ai: IBM's studio for training, validating, tuning, and deploying foundation models (LLMs). This notebook uses watsonx.ai to access to various LLMs, e.g. Llama, Granite, Ministral.

In [11]:
from google.colab import userdata
import os
os.environ["WATSONX_APIKEY"] = userdata.get('WATSONX_APIKEY')
os.environ["WATSONX_PROJECT_ID"] = userdata.get('WATSONX_PROJECT_ID')
os.environ["WATSONX_URL"] = "https://us-south.ml.cloud.ibm.com"

In [ ]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

## Initiate a LLM call

In [12]:
# Model parameter configurations
parameters = {
    "decoding_method": "greedy",  # or "sample" for creative outputs
    "max_new_tokens": 1000,
    "min_new_tokens": 1,
    "temperature": 0.0, # Controls randomness (0-2)
    "top_k": 50,  # Limits vocabulary to top k tokens
    "top_p": 1, # Nucleus sampling threshold
}

We use Llama-4-Maverick-17B-128E-Instruct-FP8, Meta's latest open-source AI model released in April 2025.

In [13]:
watsonx_llm = WatsonxLLM(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
    url=os.environ["WATSONX_URL"],
    project_id=os.environ["WATSONX_PROJECT_ID"],
    params=parameters,
)

In [14]:
response = watsonx_llm.invoke("What is the square root of 144?")
print(response)

 | HIX Tutor
The square root of 144 is 12.
#sqrt(144) = 12#
#12^2 = 144#
The square root of 144 is 12, because 12 multiplied by 12 equals 144.
- How do you simplify #sqrt(2/3) + sqrt(4/3)#?
How do you simplify #sqrt(2/3)*sqrt(3/8)#?
How do you simplify #sqrt(2/6)#?
How do you simplify #sqrt(2) + sqrt(1/2)#? 
How do you simplify #sqrt(2/3) +sqrt(1/2) #? 


## Create a LLM (ReAct) agent with a simple custom tool
A Large Language Model (LLM) agent is an intelligent system that uses an LLM not just for generating text, but also for deciding when and how to use external tools. Instead of trying to solve every problem directly, the agent interprets a user’s request, chooses the right tool (like a calculator, database, or API), and combines the results into a useful response. This makes LLM agents more reliable and practical than standalone models, since they can extend their capabilities with specialized functions.

In [16]:
# Instead of langchain.agents, use langchain_classic.agents
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_core.tools import Tool

# The rest of your code (math, re, initialize_agent) remains the same!
import re
import math

Currently, one general consensus is that LLMs struggle with math problems. One effective way to address this is by creating an LLM agent that can use external tools to perform calculations. Here, we define a simple square root function and integrate it as a tool for the agent. This allows the LLM to delegate precise mathematical operations to the tool, improving accuracy and reliability.

In [17]:
def square_root(number: float) -> str:
    """Calculate the square root of a number."""
    if number < 0:
        return "Error: Cannot calculate square root of a negative number."
    result = math.sqrt(number)
    return f"The square root of {number} is {result}"

# Create the square root tool using lambda with regex
sqrt_tool = Tool(
    name="SquareRoot",
    func=lambda s: square_root(float(re.findall(r'-?\d+\.?\d*', s)[0])),
    description="Calculate the square root of a number. Input should be a single number."
)

In [18]:
tools = [sqrt_tool]
agent = initialize_agent(
    tools=tools,
    llm=watsonx_llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

/tmp/ipykernel_3928/2983285097.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


**ReAct** (Reason+Act) prompting is a technique that combines **reasoning** and **acting** to enable AI agents to solve complex tasks through an iterative cycle. The approach works through three repeating steps: **Thought** (the agent reasons about what it knows and what to do next), **Action** (it executes a concrete step like searching the web or calling a tool), and **Observation** (it receives and processes the results). This thought-action-observation loop continues until the agent gathers enough information to complete the task.

Using ReAct-style agents combines the LLM’s reasoning abilities with external tool usage. The agent doesn’t just blindly answer a question — it thinks step by step, decides which tool (or knowledge source) to use, executes the action, observes the result, and continues reasoning if necessary. This approach is particularly powerful for complex tasks, such as multi-step math, data queries, or planning, where the model alone might make errors.

In [19]:
print(agent.agent.llm_chain.prompt.template)

Answer the following questions as best you can. You have access to the following tools:

SquareRoot(s) - Calculate the square root of a number. Input should be a single number.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [SquareRoot]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [20]:
result = agent.run("What are the factors of 156?")

/tmp/ipykernel_3928/3646495004.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = agent.run("What are the factors of 156?")




> Entering new AgentExecutor chain...
 To find the factors of 156, I need to identify all the numbers that divide 156 without leaving a remainder. I can start by checking the square root of 156, as any factor larger than that would have a corresponding factor smaller than the square root.

Action: SquareRoot
Action Input: 156
Observation:
Observation: The square root of 156.0 is 12.489995996796797
Thought: Now that I have the square root of 156, I can check all integers up to the integer part of the square root (12) to see if they are factors of 156. I will list them out and their corresponding factors.

Thought: I will check the numbers from 1 to 12 to see if they are factors of 156.

Thought: 1 is a factor because 156 / 1 = 156.
Thought: 2 is a factor because 156 / 2 = 78.
Thought: 3 is a factor because 156 / 3 = 52.
Thought: 4 is a factor because 156 / 4 = 39.
Thought: 6 is a factor because 156 / 6 = 26.
Thought: 12 is not a factor because 156 / 12 is not an integer (it's 13).
Tho

In [21]:
print("Answer:", result)

Answer: 1, 2, 3, 4, 6, 12 is not a factor, 13, 26, 39, 52, 78, 156. The correct list is 1, 2, 3, 4, 6, 13, 26, 39, 52, 78, 156.


## Create a LLM (ReAct) agent with pre-built Langchain tool
All pre-built langchain tools can be found [here](https://python.langchain.com/docs/integrations/tools/).

Some examplar tools are Google search, Math, Yahoo Finance.

The following question demonstrates thought-action-observation loop more clearly.

Note that the built-in llm-math tool in LangChain is known to be a bit error-prone. This is mostly due to string parsing issues. LLM could accidentally include extra spaces, units, or explanation in the math expression, the evaluator can fail.  That's why many people now prefer structured tools or libraries like SymPy/NumPy wrapped as tools.

## Create a LLM (React) agent with multiple tools

In [22]:
!pip install -U ddgs wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 55.6 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=157d23fa68e698379e9a0c78e918c2079e39585148ebaf2ec1cda43eeb373c74
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [23]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ==================== TOOL 1: DuckDuckGo Search ====================
search = DuckDuckGoSearchRun()

# ==================== TOOL 2: Wikipedia ====================
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# ==================== CREATE TOOLS LIST ====================
tools = [
    Tool(
        name="WebSearch",
        func=search.run,
        description="Search the internet for all information."
    ),
    Tool(
        name="Wikipedia",
        func=wikipedia.run,
        description="Search Wikipedia for detailed historical information, biographies, scientific facts, or encyclopedic knowledge. Good for established facts and background information."
    )
]

In [24]:
agent = initialize_agent(
    tools=tools,
    llm=watsonx_llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

In [25]:
result = agent.run("who is the director of Avatar? Provide a one-paragraph biography.")



> Entering new AgentExecutor chain...
 To answer this question, I need to find out who the director of Avatar is and then provide a one-paragraph biography. I'll start by searching for the director of Avatar.

Action: WebSearch
Action Input: director of Avatar
Observation:
Observation: 4 days ago -Vela-Bailey also served as Saldaña's stunt double. In 1994, Avatar directorJames Cameronwrote an 80-page treatment for the film, drawing inspiration from science fiction books he read in his childhood, as well as from adventure novels by Edgar Rice Burroughs and H. 1 week ago -He had further success with Aliens (1986), The Abyss (1989), Terminator 2: Judgment Day (1991), and True Lies (1994), as well as the Avatar franchise (2009–present). He directed, wrote, co-produced, and co-edited the historical romance epic Titanic (1997), winning Academy Awards for Best Picture, Best Director, and Best Film Editing. Three of Cameron's films—Avatar (2009), Avatar: The Way of Water (2022) and Titanic—a

In [26]:
print("Answer:", result)

Answer: James Cameron is a Canadian film director, producer, screenwriter, and explorer, known for directing films like Avatar and Titanic. Born on August 16, 1954, in Kapuskasing, Ontario, Canada, Cameron began his career in the film industry as a model maker and special effects artist. He made his directorial debut with Piranha II: The Spawning (1981) and gained recognition with The Terminator (1984). Cameron directed several successful films, including Aliens (1986), The Abyss (1989), Terminator 2: Judgment Day (1991), and True Lies (1994). His historical epic Titanic (1997) won 11 Academy Awards and became one of the highest-grossing films of all time. Cameron's 2009 film Avatar also achieved immense commercial success and became the highest-grossing film of all time upon its release. He is known for his innovative use of special effects and 3D technology in his films.
